# Random Forest Project 

For this project we will be exploring publicly available data from [LendingClub.com](https://www.lendingclub.com/). Lending Club connects people who need money (borrowers) with people who have money (investors). Hopefully, as an investor you would want to invest in people who showed a profile of having a high probability of paying you back. We will try to create a model that will help predict this.

We will use lending data from 2007-2010 and be trying to classify and predict whether or not the borrower paid back their loan in full.

Here are what the columns represent:
* **credit.policy**: 1 if the customer meets the credit underwriting criteria of LendingClub.com, and 0 otherwise.
* **purpose**: The purpose of the loan.
* **int.rate**: The interest rate of the loan.
* **installment**: The monthly installments owed by the borrower if the loan is funded.
* **log.annual.inc**: The natural log of the self-reported annual income of the borrower.
* **dti**: The debt-to-income ratio of the borrower.
* **fico**: The FICO credit score of the borrower.
* **days.with.cr.line**: The number of days the borrower has had a credit line.
* **revol.bal**: The borrower's revolving balance.
* **revol.util**: The borrower's revolving line utilization rate.
* **inq.last.6mths**: The borrower's number of inquiries by creditors in the last 6 months.
* **delinq.2yrs**: The number of times the borrower had been 30+ days past due on a payment in the past 2 years.
* **pub.rec**: The borrower's number of derogatory public records.


# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Get the Data

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
loans = pd.read_csv('loan_data.csv')

In [ ]:
loans.info()

In [ ]:
loans.head()

In [ ]:
loans.describe()

# Exploratory Data Analysis

Let's do some data visualization using seaborn and pandas built-in plotting capabilities.

**Create a histogram of two FICO distributions on top of each other, one for each credit.policy outcome.**

In [ ]:
plt.figure(figsize=(10,6))
loans[loans['credit.policy']==1]['fico'].hist(alpha=0.5, color='blue', bins=30, label='Credit Policy=1')
loans[loans['credit.policy']==0]['fico'].hist(alpha=0.5, color='red', bins=30, label='Credit Policy=0')
plt.legend()
plt.xlabel('FICO')
plt.title('FICO Distribution by Credit Policy')
plt.show()

**Create a similar figure, except this time select by the not.fully.paid column.**

In [ ]:
plt.figure(figsize=(10,6))
loans[loans['not.fully.paid']==1]['fico'].hist(alpha=0.5, color='blue', bins=30, label='Not Fully Paid=1')
loans[loans['not.fully.paid']==0]['fico'].hist(alpha=0.5, color='red', bins=30, label='Not Fully Paid=0')
plt.legend()
plt.xlabel('FICO')
plt.title('FICO Distribution by Not Fully Paid')
plt.show()

**Create a countplot using seaborn showing the counts of loans by purpose, with the color hue defined by not.fully.paid.**

In [ ]:
plt.figure(figsize=(11,7))
sns.countplot(x='purpose', data=loans, hue='not.fully.paid', palette='Set1')
plt.title('Count of Loans by Purpose')
plt.xticks(rotation=45)
plt.show()

**Let's see the trend between FICO score and interest rate using a jointplot.**

In [ ]:
sns.jointplot(x='fico', y='int.rate', data=loans, color='purple')
plt.show()

**Create lmplots to see if the trend differed between not.fully.paid and credit.policy.**

In [ ]:
sns.lmplot(x='fico', y='int.rate', data=loans, hue='credit.policy',
           col='not.fully.paid', palette='Set1')
plt.show()

# Setting up the Data

Let's get ready to set up our data for our Random Forest Classification Model!

In [ ]:
loans.info()

## Categorical Features

The **purpose** column is categorical, so we need to transform it using dummy variables.

In [ ]:
cat_feats = ['purpose']

In [ ]:
final_data = pd.get_dummies(loans, columns=cat_feats, drop_first=True)
final_data.info()

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = final_data.drop('not.fully.paid', axis=1)
y = final_data['not.fully.paid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=101)

## Training a Decision Tree Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dtree = DecisionTreeClassifier()
dtree.fit(X_train, y_train)

## Predictions and Evaluation of Decision Tree

In [ ]:
predictions = dtree.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
print('Classification Report - Decision Tree:')
print(classification_report(y_test, predictions))

In [ ]:
print('Confusion Matrix - Decision Tree:')
print(confusion_matrix(y_test, predictions))

## Training the Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rfc = RandomForestClassifier(n_estimators=600)
rfc.fit(X_train, y_train)

## Predictions and Evaluation of Random Forest

In [ ]:
rfc_pred = rfc.predict(X_test)

In [ ]:
print('Classification Report - Random Forest:')
print(classification_report(y_test, rfc_pred))

In [ ]:
print('Confusion Matrix - Random Forest:')
print(confusion_matrix(y_test, rfc_pred))

## What performed better: Random Forest or Decision Tree?

The **Random Forest** generally performs better than the single Decision Tree.

- The Decision Tree tends to **overfit** the training data, leading to lower accuracy on unseen test data.
- The Random Forest uses **multiple trees** (600 in this case) and averages their predictions, which reduces overfitting and improves generalization.
- In terms of **accuracy**, the Random Forest typically achieves higher overall accuracy on the test set.
- However, both models may show lower **recall for class 1** (not fully paid), because the dataset is **imbalanced** — there are far fewer loans that were not fully paid compared to those that were paid.

**Conclusion:** Random Forest is the better model for this problem due to its ensemble approach and resistance to overfitting.

# Great Job!